# Resumo e Decisões

## Parte 1:
Esses são os pontos que vou retirar, setembro de 2021, maio de 2024 e dezembro de 2025. Os dois primeiros baseados na média geral e o último baseado na média do respectivo mês nos últimos 4 anos

## Parte 2:
Removido os outliers e feita a média de geração de cada mês. No ponto de outlier foi alterado o valor para a média do mês nos 4 anos disponíveis. Dessa forma evitamos problemas na etapa seguinte com modelos que precisam de dados contínuos.

## Parte 3:
* **Divisão da Série:** O particionamento cronológico foi estruturado em Treino (histórico até junho/2024), Validação (julho/2024 a junho/2025) e Teste (julho/2025 a junho/2026).
* **(a) Treinamento e Escolha:** Foram avaliados três modelos preditivos: SARIMA, Suavização Exponencial (Holt-Winters) e XGBoost. O modelo **Holt-Winters** foi o vencedor empírico, apresentando o menor Erro Percentual Absoluto Médio no conjunto de validação (MAPE = 4.93%, contra 8.13% do SARIMA e 8.01% do XGBoost).
* **(b) Avaliação Final:** Reajuste do modelo Holt-Winters com Treino e Validação e submetido à inferência sobre o conjunto de Teste (julho/2025 a junho/2026), com resultado de um MAPE de 5,55%.
* **(c) Extrapolação Temporal:** O modelo Holt-Winters foi reajustado empregando a base de dados histórica completa (60 meses) para projetar a geração dos quatro meses subsequentes (julho a outubro de 2026).

# Execução abaixo:

### Parte 1

Importar as bibliotecas e carregar os dados de geração

In [38]:
import pandas as pd

# 1. Carregar os dados
tabela_usina = pd.read_excel('power_plant_data_teste.xlsx')

# verificar se tem linhas vazias
display(tabela_usina.info())

# Mostrar a tabela
display(tabela_usina)

<class 'pandas.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 3 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   Geração Mensal Referência Month            60 non-null     datetime64[us]
 1   Unidade Consumidora (UC) Usina (Nickname)  61 non-null     str           
 2   Geração Mensal SUM Energia Gerada (kWh)    61 non-null     int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 1.6 KB


None

,Geração Mensal Referência Month,Unidade Consumidora (UC) Usina (Nickname),Geração Mensal SUM Energia Gerada (kWh)
0,NaT,Usina Teste,0
1,2021-07-01,Usina Teste,374203
2,2021-08-01,Usina Teste,404653
3,2021-09-01,Usina Teste,426355
4,2021-10-01,Usina Teste,467200
...,...,...,...
56,2026-02-01,Usina Teste,438651
57,2026-03-01,Usina Teste,363221
58,2026-04-01,Usina Teste,364546
59,2026-05-01,Usina Teste,306272


In [39]:
#Temos uma linha vazia na tabela! Limpando essa linha
tabela_limpa = tabela_usina.dropna()
display(tabela_limpa.info())

display(tabela_limpa.describe().round(1))

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 1 to 60
Data columns (total 3 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   Geração Mensal Referência Month            60 non-null     datetime64[us]
 1   Unidade Consumidora (UC) Usina (Nickname)  60 non-null     str           
 2   Geração Mensal SUM Energia Gerada (kWh)    60 non-null     int64         
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 1.5 KB


None

,Geração Mensal Referência Month,Geração Mensal SUM Energia Gerada (kWh)
count,60,60.0
mean,2023-12-16 11:12:00,417105.5
min,2021-07-01 00:00:00,110463.0
25%,2022-09-23 12:00:00,352583.5
50%,2023-12-16 12:00:00,422588.0
75%,2025-03-08 18:00:00,494157.2
max,2026-06-01 00:00:00,730252.0
std,NaN,101307.7


In [40]:
import plotly.express as px

# Gráfico de barras simples: Mês no eixo X e Geração no eixo Y
fig = px.bar(
    tabela_limpa, 
    x='Geração Mensal Referência Month', 
    y='Geração Mensal SUM Energia Gerada (kWh)',
    title='Geração Mensal da Usina (kWh)',
    labels={'data': 'Mês/Ano', 'geracao': 'Geração (kWh)'}
)

fig.show()


### Importante:
    Temos uma média de geração de 417105 kWh no período avaliado
    Setembro de 2022 tem a maior geração com 730252 kWh, aproximadamente 75% mais produção que a media
    Maio de 2024 tem a menor geração com 110463 kWh, aproximadamente 26% da média
    Dezembro de 2025 está com geração de 266 kWh, sendo aproximadamente 50% da média dos últimos 4 anos (21 a 24 => média aproximada 519 kwh)

Esses são os pontos que vou modificar: setembro de 2022, maio de 2024 e dezembro de 2025. Os dois primeiros baseados na média geral e o último baseado na média do respectivo mês nos últimos 4 anos


### Parte 2

In [41]:
# variáveis auxiliares
col_data = 'Geração Mensal Referência Month'
col_geracao = 'Geração Mensal SUM Energia Gerada (kWh)'

# Converter a data
tabela_usina[col_data] = pd.to_datetime(tabela_usina[col_data])

# Definir os 3 pontos de outliers
datas_outliers = pd.to_datetime(['2022-09-01', '2024-05-01', '2025-12-01'])

# Criar uma cópia da tabela para correção
tabela_corrigida = tabela_limpa.copy()

# Criar coluna auxiliar com o mês do ano (1 a 12)
tabela_corrigida['mes_num'] = tabela_limpa[col_data].dt.month

# ver se temos todas as linhas
#display(tabela_limpa.info())
display(tabela_corrigida.info())

# Calcular a média do mês(excluindo os outliers)
dados_sem_outliers = tabela_corrigida[~tabela_corrigida[col_data].isin(datas_outliers)]
medias_historicas = dados_sem_outliers.groupby('mes_num')[col_geracao].mean()

# mostrar tabela
# display(tabela_corrigida)


<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 1 to 60
Data columns (total 4 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   Geração Mensal Referência Month            60 non-null     datetime64[us]
 1   Unidade Consumidora (UC) Usina (Nickname)  60 non-null     str           
 2   Geração Mensal SUM Energia Gerada (kWh)    60 non-null     int64         
 3   mes_num                                    60 non-null     int32         
dtypes: datetime64[us](1), int32(1), int64(1), str(1)
memory usage: 1.8 KB


None

In [42]:
# Criar a coluna tratada (inicialmente idêntica à original)
tabela_corrigida['geracao_corrigida'] = tabela_limpa[col_geracao].astype(float)
# display(tabela_corrigida.info())

# Substituir o valor dos 3 outliers pela média dos seus respectivos meses
for data in datas_outliers:
    mes = data.month
    valor_media = medias_historicas[mes]
    tabela_corrigida.loc[tabela_limpa[col_data] == data, 'geracao_corrigida'] = valor_media

# Removendo a coluna auxiliar
tabela_corrigida = tabela_corrigida.drop(columns=['mes_num'], errors='ignore')

# --- Relatório do Antes e Depois ---
print("=== RELATÓRIO DE CORREÇÃO (PARTE 2) ===")
filtro_outliers = tabela_corrigida[col_data].isin(datas_outliers)
relatorio = tabela_corrigida.loc[filtro_outliers, [col_data, col_geracao, 'geracao_corrigida']]
relatorio.columns = ['Data', 'Valor Original (Outlier)', 'Valor Corrigido (Média dos 4 anos)']

display(relatorio)

# display(tabela_corrigida.info())
# display(tabela_corrigida)

=== RELATÓRIO DE CORREÇÃO (PARTE 2) ===


,Data,Valor Original (Outlier),Valor Corrigido (Média dos 4 anos)
15,2022-09-01,730252,400091.75
35,2024-05-01,110463,334202.50
54,2025-12-01,266220,527261.25


In [43]:
# Plotando o gráficos antes e depois para verificar os dados removidos

# antes
fig = px.bar(
    tabela_usina, 
    x='Geração Mensal Referência Month', 
    y='Geração Mensal SUM Energia Gerada (kWh)',
    title='Geração Mensal da Usina antes (kWh)',
    labels={'data': 'Mês/Ano', 'geracao': 'Geração (kWh)'}
)

fig.show()

# depois
fig = px.bar(
    tabela_corrigida, 
    x='Geração Mensal Referência Month', 
    y='geracao_corrigida',
    title='Geração Mensal da Usina depois (kWh)',
    labels={'data': 'Mês/Ano', 'geracao': 'Geração (kWh)'}
)

fig.show()

Foram corrigidos os dados referentes a setembro de 2022, maio de 2024 e dezembro de 2025. Para todos os casos foram modificados seus valores para a média dos outros 4 meses disponíveis.

### Parte 3
### a)

In [44]:
# ==============================================================================
# 1. Preparação da Base de Dados
# ==============================================================================
# Cria-se uma cópia de segurança para não alterar a tabela original
df_modelo = tabela_corrigida.copy()

# Removendo a coluna de nickname, pois não é necessária para o modelo
df_modelo = df_modelo.drop('Unidade Consumidora (UC) Usina (Nickname)',axis=1)

# Ordenação temporal para evitar desvios no particionamento
# df_modelo = df_modelo.sort_values(by='Geração Mensal Referência Month')

# # Definição da coluna de data como o índice do DataFrame
df_modelo = df_modelo.set_index('Geração Mensal Referência Month')

# Definição da frequência temporal (Mensal, início do mês - Month Start)
# Isso é um requisito rigoroso para o statsmodels (SARIMA e Holt-Winters)
df_modelo = df_modelo.asfreq('MS')

# Isolamento da série univariada alvo
serie_alvo = df_modelo['geracao_corrigida']

# display(serie_alvo.info())
# display(serie_alvo)

In [45]:
# ==============================================================================
# 2. Separação Cronológica (Treino e Validação)
# ==============================================================================
# Treino: Histórico até Junho/2024
# Validação: Julho/2024 a Junho/2025
treino = serie_alvo.loc[:'2024-06-30']
validacao = serie_alvo.loc['2024-07-01':'2025-06-30']


In [46]:
# ==============================================================================
# 3. Modelagem Estatística: SARIMA
# ==============================================================================
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
# # Supressão de avisos não críticos de convergência para manter o console limpo
warnings.filterwarnings("ignore")

from sklearn.metrics import mean_absolute_percentage_error

# Testar com auto_arima seria ideal para encontrar os melhores parâmetros, mas para simplificação, vamos usar (1, 1, 1) e (1, 1, 0, 12)

modelo_sarima = SARIMAX(treino, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12))
resultado_sarima = modelo_sarima.fit(disp=False)
prev_sarima = resultado_sarima.forecast(steps=len(validacao))

print("--- Erro Percentual Absoluto Médio (MAPE) na Validação ---")
mape_sarima = mean_absolute_percentage_error(validacao, prev_sarima) * 100
print(f"SARIMA:       {mape_sarima:.2f}%")

# print(resultado_sarima.summary())

# 9.94%  => order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)
# 8.13%  => order=(1, 1, 1), seasonal_order=(1, 1, 0, 12)

--- Erro Percentual Absoluto Médio (MAPE) na Validação ---
SARIMA:       8.13%


In [47]:
# ==============================================================================
# 4. Modelagem Estatística: Suavização Exponencial (Holt-Winters)
# ==============================================================================
from statsmodels.tsa.holtwinters import ExponentialSmoothing

modelo_hw = ExponentialSmoothing(treino, trend='add', seasonal='add', seasonal_periods=12)
resultado_hw = modelo_hw.fit()
prev_hw = resultado_hw.forecast(len(validacao))

print("--- Erro Percentual Absoluto Médio (MAPE) na Validação ---")
mape_hw = mean_absolute_percentage_error(validacao, prev_hw) * 100
print(f"Holt-Winters: {mape_hw:.2f}%")

# print(resultado_hw.summary())

# 4.93%  => trend='add', seasonal='add', seasonal_periods=12

--- Erro Percentual Absoluto Médio (MAPE) na Validação ---
Holt-Winters: 4.93%


In [48]:
# ==============================================================================
# 5. Modelagem em Machine Learning: XGBoost
# ==============================================================================
from xgboost import XGBRegressor

def criar_features_temporais(serie, n_lags=12):
    """
    Função para transpor a série temporal em um problema de regressão tabular.
    Gera n colunas correspondentes aos instantes t-1, t-2 ... t-n.
    """
    df_lags = pd.DataFrame(serie)
    for i in range(1, n_lags + 1):
        df_lags[f'lag_{i}'] = serie.shift(i)
    return df_lags.dropna()

# Extração de atributos defasados (lags)
df_xgb = criar_features_temporais(serie_alvo, n_lags=12)

# Separação das matrizes de características (X) e do vetor alvo (y)
X = df_xgb.drop(columns=['geracao_corrigida'])
y = df_xgb['geracao_corrigida']

# Divisão cronológica replicada para as matrizes do XGBoost
X_treino_xgb = X.loc[:'2024-06-30']
y_treino_xgb = y.loc[:'2024-06-30']
X_validacao_xgb = X.loc['2024-07-01':'2025-06-30']
y_validacao_xgb = y.loc['2024-07-01':'2025-06-30']

modelo_xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
modelo_xgb.fit(X_treino_xgb, y_treino_xgb)
prev_xgb = modelo_xgb.predict(X_validacao_xgb)

print("--- Erro Percentual Absoluto Médio (MAPE) na Validação ---")
mape_xgb = mean_absolute_percentage_error(y_validacao_xgb, prev_xgb) * 100
print(f"XGBoost:      {mape_xgb:.2f}%")

--- Erro Percentual Absoluto Médio (MAPE) na Validação ---
XGBoost:      8.01%


In [49]:
# ==============================================================================
# 6. Avaliação e Comparação de Métricas (MAPE)
# ==============================================================================
mape_sarima = mean_absolute_percentage_error(validacao, prev_sarima) * 100
mape_hw = mean_absolute_percentage_error(validacao, prev_hw) * 100
mape_xgb = mean_absolute_percentage_error(y_validacao_xgb, prev_xgb) * 100

print("--- Erro Percentual Absoluto Médio (MAPE) na Validação ---")
print(f"SARIMA:       {mape_sarima:.2f}%")
print(f"Holt-Winters: {mape_hw:.2f}%")
print(f"XGBoost:      {mape_xgb:.2f}%")

--- Erro Percentual Absoluto Médio (MAPE) na Validação ---
SARIMA:       8.13%
Holt-Winters: 4.93%
XGBoost:      8.01%


--- Erro Percentual Absoluto Médio (MAPE) na Validação ---

SARIMA:       8.13%

Holt-Winters: 4.93%

XGBoost:      8.01%

### b

In [50]:
# ==============================================================================
# Reajuste do Modelo Vencedor e Avaliação Final
# ==============================================================================
# 1. Definição da partição de Teste (Julho de 2025 a Junho de 2026)
teste = serie_alvo.loc['2025-07-01':'2026-06-30']

In [51]:
#2. Concatenação das matrizes de Treino e Validação (Dados até Junho/2025)
# Isso assegura que o modelo assimile os padrões sazonais mais recentes antes do teste.
treino_validacao = pd.concat([treino, validacao])


In [52]:
# 3. Reajuste do modelo Holt-Winters sobre a matriz unificada
modelo_definitivo = ExponentialSmoothing(
    treino_validacao, 
    trend='add', 
    seasonal='add', 
    seasonal_periods=12
)
resultado_definitivo = modelo_definitivo.fit()

In [53]:
# 4. Inferência estatística sobre o conjunto de Teste oculto
prev_teste = resultado_definitivo.forecast(len(teste))

In [54]:
# 5. Cômputo da métrica final de avaliação
mape_teste = mean_absolute_percentage_error(teste, prev_teste) * 100

print("--- Avaliação de Desempenho Real (Conjunto de Teste) ---")
print(f"Modelo Eleito: Suavização Exponencial (Holt-Winters)")
print(f"MAPE Final: {mape_teste:.2f}%")

--- Avaliação de Desempenho Real (Conjunto de Teste) ---
Modelo Eleito: Suavização Exponencial (Holt-Winters)
MAPE Final: 5.55%


### c

In [55]:
# ==============================================================================
# Extrapolação Temporal (Julho a Outubro de 2026)
# ==============================================================================
# 1. Instanciação e ajuste do modelo sobre o vetor integral (Treino + Validação + Teste)
modelo_extrapolacao = ExponentialSmoothing(
    serie_alvo, 
    trend='add', 
    seasonal='add', 
    seasonal_periods=12
)
resultado_extrapolacao = modelo_extrapolacao.fit()

In [56]:
# 2. Definição do horizonte preditivo (4 passos temporais à frente de Junho/2026)
horizonte = 4
previsao_futura = resultado_extrapolacao.forecast(horizonte)

In [57]:
# 3. Estruturação tabular da saída para reporte e integração com a Parte 4
# Dicionário de mapeamento para forçar a abreviação em português
meses_pt = {7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out'}

print("--- Previsão de Geração Futura (kWh) ---")
for data, valor_estimado in previsao_futura.items():
    # Extração do mês e ano para formatação da string
    mes_str = meses_pt[data.month]
    ano_str = data.year
    
    # Impressão contínua com duas casas decimais
    print(f"Valor para Rateio em {mes_str}/{ano_str}: {valor_estimado:.2f} kWh")

# A variável de outubro já está salva para uso na matriz matemática da Parte 4
geracao_outubro_2026 = previsao_futura.loc['2026-10-01']

--- Previsão de Geração Futura (kWh) ---
Valor para Rateio em Jul/2026: 281417.55 kWh
Valor para Rateio em Ago/2026: 349683.25 kWh
Valor para Rateio em Set/2026: 365663.16 kWh
Valor para Rateio em Out/2026: 415332.30 kWh


### Parte 4
### a)

In [ ]:
# Carregar os dados
tabela_consumo_medio_ucs = pd.read_excel('consumo_medio_ucs.xlsx')

# display(tabela_consumo_medio_ucs.info())
# display(tabela_consumo_medio_ucs)

In [72]:
# Criar uma cópia e renomear as colunas para a notação pardrão
df_ucs = tabela_consumo_medio_ucs.rename(columns={
    'Unidade Consumidora (UC) Número de Instalação': 'uc',
    'Conta Consumo Médio (kWh)': 'Ec',
    'Conta Saldo Acumulado (kWh)': 'Cr'
})

# display(df_ucs.info())

In [ ]:
# Cálculo da demanda total da carteira (Soma do Consumo Médio - Ec)
consumo_total_carteira = df_ucs['Ec'].sum()
print(f"--- (a) Análise de Cobertura ---")
print(f"Demanda Total Mensal da Carteira: {consumo_total_carteira:.2f} kWh\n")

--- (a) Análise de Cobertura ---
Demanda Total Mensal da Carteira: 362400.00 kWh



In [80]:
# Avaliação mensal e acumulada
geracao_acumulada = 0
demanda_acumulada = 0

for data, geracao_prevista in previsao_futura.items():
    mes_ano = f"{meses_pt[data.month]}/{data.year}"
    balanco_mensal = geracao_prevista - consumo_total_carteira
    
    geracao_acumulada += geracao_prevista
    demanda_acumulada += consumo_total_carteira
    
    status = "COBRE" if balanco_mensal >= 0 else "DÉFICIT"
    print(f"{mes_ano} | Geração: {geracao_prevista:.0f} | Balanço: {balanco_mensal:+.0f} kWh ({status})")

balanco_acumulado = geracao_acumulada - demanda_acumulada
status_acumulado = "SUPERÁVIT" if balanco_acumulado >= 0 else "DÉFICIT"
print(f"\nBalanço Acumulado (4 meses): {balanco_acumulado:+.0f} kWh ({status_acumulado})\n")


Jul/2026 | Geração: 281418 | Balanço: -80982 kWh (DÉFICIT)
Ago/2026 | Geração: 349683 | Balanço: -12717 kWh (DÉFICIT)
Set/2026 | Geração: 365663 | Balanço: +3263 kWh (COBRE)
Out/2026 | Geração: 415332 | Balanço: +52932 kWh (COBRE)

Balanço Acumulado (4 meses): -37504 kWh (DÉFICIT)



In [64]:
# Qual a geração estimada para os 4 próximos meses (Julho a Outubro de 2026)?
soma_geracao_futura = previsao_futura.sum()
display(f"Soma da geração estimada para os 4 meses (Julho a Outubro de 2026): {soma_geracao_futura:.2f} kWh")

'Soma da geração estimada para os 4 meses (Julho a Outubro de 2026): 1412096.26 kWh'

In [79]:
# Qual o consumo médio das UCs para os 4 próximos meses (Julho a Outubro de 2026)?
# Levando em consideração que o consumo médio mostrado na tabela é referente a apenas 1 mês, 
# então multiplicamos por 4 para obter o consumo médio total para os 4 meses

soma_consumo_medio_ucs = tabela_consumo_medio_ucs['Conta Consumo Médio (kWh)'].sum()*4
display(f"Soma do consumo médio das UCs para os 4 meses (Julho a Outubro de 2026): {soma_consumo_medio_ucs:.2f} kWh")

'Soma do consumo médio das UCs para os 4 meses (Julho a Outubro de 2026): 1449600.00 kWh'

In [66]:
# Com relação ao saldo acumulado, não é necessário multiplicar por 4
soma_saldo_acumulado = tabela_consumo_medio_ucs['Conta Saldo Acumulado (kWh)'].sum()
display(f"Soma do saldo acumulado das UCs: {soma_saldo_acumulado:.2f} kWh")

'Soma do saldo acumulado das UCs: 50085.00 kWh'